In [ ]:
# ==========================================
# PASO 1: IDENTIFICACIÓN DE LA FUENTE DE DATOS
# ==========================================
organismo_fuente = "Consejería de Fomento y Movilidad (Junta de Andalucía)"
licencia_uso = "Reconocimiento 4.0 Internacional (CC BY 4.0)"
nombre_dataset = "Puntos de venta del Consorcio de Transportes de la Bahía de Cádiz"
url_dataset_web = "https://www.juntadeandalucia.es/datosabiertos/portal/dataset/consorcio-de-transporte-bahia-de-cadiz-puntos-de-venta"
url_csv_directo = "https://www.juntadeandalucia.es/datosabiertos/portal/dataset/8555d8b2-5688-4481-bf8c-733020dfc484/resource/300b6ba5-c04d-42c3-8185-687c5d5a8314/download/puntos-de-venta-consorcio-de-transporte-cadiz.csv"

print("=== FICHA DE FUENTE DE DATOS ===")
print(f"• Dataset: {nombre_dataset}")
print(f"• Organismo Público: {organismo_fuente}")
print(f"• Licencia de Uso: {licencia_uso}")
print(f"• URL del Portal: {url_dataset_web}")
print(f"• URL de Descarga Directa CSV: {url_csv_directo}")


# ==========================================
# PASO 2: CARGA DE DATOS Y EXPLORACIÓN INICIAL
# ==========================================
import pandas as pd

url_csv = "https://www.juntadeandalucia.es/datosabiertos/portal/dataset/8555d8b2-5688-4481-bf8c-733020dfc484/resource/300b6ba5-c04d-42c3-8185-687c5d5a8314/download/puntos-de-venta-consorcio-de-transporte-cadiz.csv"

df = pd.read_csv(url_csv, sep=";", encoding="latin-1", skiprows=1)

print("\n--- 1. PRIMERAS FILAS DEL DATAFRAME ---")
display(df.head())

print("\n--- 2. DIMENSIONES (Filas, Columnas) ---")
print(df.shape)

print("\n--- 3. TIPOS DE DATOS INICIALES ---")
print(df.dtypes)

print("\n--- 4. ANÁLISIS DE VALORES NULOS ---")
nulos_abs = df.isna().sum()
nulos_porcentual = (df.isna().mean() * 100).round(2)

df_nulos = pd.DataFrame({
    'Nulos Absolutos': nulos_abs,
    'Porcentaje (%)': nulos_porcentual
})
print(df_nulos)


# ==========================================
# PASO 3: TRANSFORMACIONES DE DATOS
# ==========================================

# 1. Transformación 1: Limpieza de registros y metadatos del footer
df = df.dropna(subset=['TIPO']).copy()

# 2. Transformación 2: Normalización de coordenadas geográficas (regex y conversión)
import re

mask = (df['LATITUD'].str.match(r'-?\d{7,8}$', na=False)) & (df['LONGITUD'].str.match(r'-?\d{7,8}$', na=False))

df.loc[mask, 'LATITUD'] = df.loc[mask, 'LATITUD'].astype(float) / 10**6
df.loc[mask, 'LONGITUD'] = df.loc[mask, 'LONGITUD'].astype(float) / 10**6
df.loc[~mask, 'LATITUD'] = None
df.loc[~mask, 'LONGITUD'] = None

df['LATITUD'] = pd.to_numeric(df['LATITUD'])
df['LONGITUD'] = pd.to_numeric(df['LONGITUD'])

# 3. Transformación 3: Agrupación y Resumen por Tipo de Establecimiento
resumen_tipos = df['TIPO'].value_counts()
print("\n--- RESUMEN: PUNTOS DE VENTA POR TIPO ---")
print(resumen_tipos)


# ==========================================
# PASO 4 (EXTRA): VISUALIZACIÓN ESPACIAL (MAPA)
# ==========================================
import plotly.express as px

df_grafica = df.dropna(subset=['LATITUD', 'LONGITUD']).copy()

fig = px.scatter_mapbox(
    df_grafica,
    lat="LATITUD",
    lon="LONGITUD",
    hover_name="DIRECCIÓN",
    hover_data=["TIPO", "MUNICIPIO"],
    zoom=9,
    height=600
)

carto_url_con_key = "https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_4lf2_1_1861c94fce88c694b807afa5"

fig.update_layout(
    mapbox=dict(
        style="white-bg",
        center=dict(
            lat=df_grafica["LATITUD"].mean(),
            lon=df_grafica["LONGITUD"].mean()
        ),
        layers=[
            {
                "below": 'traces',
                "sourcetype": "raster",
                "source": [carto_url_con_key]
            }
        ]
    ),
    margin={"r":0, "t":0, "l":0, "b":0}
)

fig.show(renderer="colab")

=== FICHA DE FUENTE DE DATOS ===
• Dataset: Puntos de venta del Consorcio de Transportes de la Bahía de Cádiz
• Organismo Público: Consejería de Fomento y Movilidad (Junta de Andalucía)
• Licencia de Uso: Reconocimiento 4.0 Internacional (CC BY 4.0)
• URL del Portal: https://www.juntadeandalucia.es/datosabiertos/portal/dataset/consorcio-de-transporte-bahia-de-cadiz-puntos-de-venta
• URL de Descarga Directa CSV: https://www.juntadeandalucia.es/datosabiertos/portal/dataset/8555d8b2-5688-4481-bf8c-733020dfc484/resource/300b6ba5-c04d-42c3-8185-687c5d5a8314/download/puntos-de-venta-consorcio-de-transporte-cadiz.csv

--- 1. PRIMERAS FILAS DEL DATAFRAME ---


,TIPO,MUNICIPIO,NÚCLEO,DIRECCIÓN,LATITUD,LONGITUD,URL
0,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Urb. Espíritu Santo, L 25-26",36774299,-6371899,http://apirtod.dipucadiz.es/api/datos/puntos_v...
1,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"PÂº. Jardín Botánico, portal 2 (p-3)",36769548,-6354767,http://apirtod.dipucadiz.es/api/datos/puntos_v...
2,Comercio,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Plaza JaÃ©n, 1",36795763,-6337953,http://apirtod.dipucadiz.es/api/datos/puntos_v...
3,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Cl. Sebastián Elcano, 11","3,68E+15","-6,36E+15",http://apirtod.dipucadiz.es/api/datos/puntos_v...
4,Estanco,Sanlucar de Barrameda,Sanlúcar de Barrameda,"Cl. Santo Domingo, 90","3,68E+15","-6,35E+15",http://apirtod.dipucadiz.es/api/datos/puntos_v...



--- 2. DIMENSIONES (Filas, Columnas) ---
(247, 7)

--- 3. TIPOS DE DATOS INICIALES ---
TIPO         object
MUNICIPIO    object
NÚCLEO       object
DIRECCIÓN    object
LATITUD      object
LONGITUD     object
URL          object
dtype: object

--- 4. ANÁLISIS DE VALORES NULOS ---
           Nulos Absolutos  Porcentaje (%)
TIPO                     2            0.81
MUNICIPIO                3            1.21
NÚCLEO                   3            1.21
DIRECCIÓN                3            1.21
LATITUD                  3            1.21
LONGITUD                 3            1.21
URL                      3            1.21

--- RESUMEN: PUNTOS DE VENTA POR TIPO ---
TIPO
Estanco                                                                                                    170
Comercio                                                                                                    45
Taquilla                                                                                                  